In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install mediapipe

In [ ]:
import mediapipe as mp
import sys

print(f"Python Version: {sys.version}")
print(f"MediaPipe Version: {mp.__version__}")

# Check for the Legacy API (the one we want)
if hasattr(mp, 'solutions') and hasattr(mp.solutions, 'face_mesh'):
    print("✅ STATUS: Legacy API is AVAILABLE. You can use FaceMesh.")
    COMPATIBLE = "LEGACY"
else:
    print("❌ STATUS: Legacy API is MISSING. You are likely on a newer version.")
    COMPATIBLE = "TASKS"

# Check if Face Mesh works specifically
try:
    from mediapipe.python.solutions import face_mesh as fm
    print("✅ FaceMesh module located.")
except ImportError:
    print("⚠️ FaceMesh module not found in standard path.")

In [ ]:
import cv2
import mediapipe as mp
import os
import numpy as np
from pathlib import Path

# 1. Initialize FaceMesh
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(
    static_image_mode=True, 
    max_num_faces=1, 
    refine_landmarks=True,
    min_detection_confidence=0.5
)

# 2. Folder Paths
input_root = "/kaggle/input/autistic-and-normal-image"
output_folder = "/kaggle/working/ssl_eye_dataset"
os.makedirs(output_folder, exist_ok=True)


# Landmarks for eyes (from MediaPipe documentation)
LEFT_EYE = [33, 7, 163, 144, 145, 153, 154, 155, 133, 173, 157, 158, 159, 160, 161, 246]
RIGHT_EYE = [362, 382, 381, 380, 374, 373, 390, 249, 263, 466, 388, 387, 386, 385, 384, 398]

def get_eye_crop(image, landmarks, points, padding=0.5):
    h, w, _ = image.shape
    x_pts = [int(landmarks[p].x * w) for p in points]
    y_pts = [int(landmarks[p].y * h) for p in points]
    
    xmin, xmax = min(x_pts), max(x_pts)
    ymin, ymax = min(y_pts), max(y_pts)
    
    # Calculate box size with extra padding for low-res context
    eye_w, eye_h = xmax - xmin, ymax - ymin
    side = max(eye_w, eye_h) * (1 + padding)
    
    # Find center
    cx, cy = (xmin + xmax) // 2, (ymin + ymax) // 2
    
    # Define square coordinates
    x1, x2 = int(cx - side//2), int(cx + side//2)
    y1, y2 = int(cy - side//2), int(cy + side//2)
    
    # Crop and ensure it stays within image boundaries
    crop = image[max(0, y1):min(h, y2), max(0, x1):min(w, x2)]
    return crop

# 3. Processing Loop
print("🚀 Starting extraction... this might take a few minutes.")
count = 0
error_count = 0

# rglob finds all jpg files in any subfolder (Train, Test, Valid)
for path in Path(input_root).rglob('*.jpg'):
    img = cv2.imread(str(path))
    if img is None:
        continue
    
    # Convert BGR to RGB for MediaPipe
    rgb_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(rgb_img)
    
    if results.multi_face_landmarks:
        landmarks = results.multi_face_landmarks[0].landmark
        
        # We create a unique name based on the path to avoid overwriting "001.jpg"
        # Example: ASD_Data_Train_autism_0001_L.jpg
        unique_prefix = "_".join(path.parts[path.parts.index('autistic-and-normal-image')+1:-1])
        
        for side, points in [("L", LEFT_EYE), ("R", RIGHT_EYE)]:
            eye_img = get_eye_crop(img, landmarks, points)
            
            if eye_img.size > 0:
                # Resize to 300x300 for the Transformer model
                final_eye = cv2.resize(eye_img, (300, 300))
                
                save_name = f"{unique_prefix}_{path.stem}_{side}.jpg"
                cv2.imwrite(os.path.join(output_folder, save_name), final_eye)
                count += 1
    else:
        error_count += 1

print(f"\n✅ FINISHED!")
print(f"Total Eyes Extracted: {count}")
print(f"Faces not detected in {error_count} images.")
print(f"Location: {output_folder}")

In [ ]:
import shutil
import os

# Define the folder you want to zip and the name of the output zip file
folder_to_zip = '/kaggle/working/ssl_eye_dataset'
output_zip = '/kaggle/working/ssl_eye_dataset'

# Create the zip file
if os.path.exists(folder_to_zip):
    shutil.make_archive(output_zip, 'zip', folder_to_zip)
    print(f"✅ Success! Your dataset has been zipped: {output_zip}.zip")
    print("You can now download it from the 'Output' section on the right sidebar.")
else:
    print(f"❌ Error: The folder {folder_to_zip} was not found. Did the cropping script finish?")

In [ ]:
import cv2
import mediapipe as mp
import os
from pathlib import Path

# 1. Setup Aggressive MediaPipe
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(static_image_mode=True, max_num_faces=1, min_detection_confidence=0.3)

input_root = "/kaggle/input/et-autism-dataset"
output_folder = "/kaggle/working/ssl_severity_dataset"
os.makedirs(output_folder, exist_ok=True)

LEFT_EYE = [33, 133, 159, 145]
RIGHT_EYE = [362, 263, 386, 374]

def get_crop(image, landmarks, points, padding=0.5):
    h, w, _ = image.shape
    x_pts = [int(landmarks[p].x * w) for p in points]
    y_pts = [int(landmarks[p].y * h) for p in points]
    side = max(max(x_pts)-min(x_pts), max(y_pts)-min(y_pts)) * (1 + padding)
    cx, cy = (min(x_pts)+max(x_pts))//2, (min(y_pts)+max(y_pts))//2
    return image[max(0, int(cy-side//2)):min(h, int(cy+side//2)), 
                 max(0, int(cx-side//2)):min(w, int(cx+side//2))]

# 2. Unified Loop
for cat in ['high', 'low', 'medium', 'mild']:
    save_path = Path(output_folder) / cat
    save_path.mkdir(parents=True, exist_ok=True)
    files = list((Path(input_root)/cat).rglob('*.jpg'))
    count = 0

    for img_path in files:
        img = cv2.imread(str(img_path))
        if img is None: continue
        
        results = face_mesh.process(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        
        # Try MediaPipe First (For alignment with SSL data)
        if results.multi_face_landmarks:
            mesh = results.multi_face_landmarks[0].landmark
            side = "le" if "le" in img_path.name.lower() else "re"
            pts = LEFT_EYE if side == "le" else RIGHT_EYE
            crop = get_crop(img, mesh, pts)
        else:
            # Fallback: Your original logic but with 0.5 padding feel
            h, w, _ = img.shape
            s = int(min(h, w) * 0.45) # Simulates the 0.5 padding look
            crop = img[(h//2)-s:(h//2)+s, (w//2)-s:(w//2)+s]

        if crop.size > 0:
            cv2.imwrite(str(save_path / img_path.name), cv2.resize(crop, (300, 300)))
            count += 1
    print(f"✅ {cat.upper()}: {count} images aligned.")

In [2]:
import shutil
import os

# Define the folder you want to zip and the name of the output zip file
folder_to_zip = '/kaggle/working/ssl_severity_dataset'
output_zip = '/kaggle/working/ssl_severity_dataset'

# Create the zip file
if os.path.exists(folder_to_zip):
    shutil.make_archive(output_zip, 'zip', folder_to_zip)
    print(f"✅ Success! Your dataset has been zipped: {output_zip}.zip")
    print("You can now download it from the 'Output' section on the right sidebar.")
else:
    print(f"❌ Error: The folder {folder_to_zip} was not found. Did the cropping script finish?")

KeyboardInterrupt: 

In [7]:
import cv2
import os
import numpy as np
from pathlib import Path

def apply_medical_sharpening(img):
    # 1. Noise Suppression (Bilateral Filter)
    img = cv2.bilateralFilter(img, d=5, sigmaColor=75, sigmaSpace=75)
    
    # 2. Targeted Contrast Enhancement in LAB Space
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(4,4))
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    cl = clahe.apply(l)
    
    limg = cv2.merge((cl, a, b))
    return cv2.cvtColor(limg, cv2.COLOR_LAB2BGR)

# --- CORRECTED PATHS FOR KAGGLE ---
# These now match your 'Notebook Input' sidebar
ssl_input_general = Path("/kaggle/input/ssl-dataset/ssl_eye_dataset")
ssl_input_severity = Path("/kaggle/input/ssl-dataset/ssl_severity_dataset")

# Output directory in your working space
final_output_base = Path("/kaggle/working/final_ssl_aligned")

# --- PROCESS GENERAL KNOWLEDGE DATASET ---
general_output = final_output_base / "general"
general_output.mkdir(parents=True, exist_ok=True)

general_files = list(ssl_input_general.glob('*.jpg'))
print(f"🚀 Sharpening {len(general_files)} General Knowledge images...")
for i, f in enumerate(general_files):
    img = cv2.imread(str(f))
    if img is not None:
        cv2.imwrite(str(general_output / f.name), apply_medical_sharpening(img))
    if (i+1) % 1000 == 0: print(f"Done: {i+1}")

# --- PROCESS SEVERITY DATASET ---
severity_output = final_output_base / "severity"

print(f"\n🚀 Sharpening Severity categories...")
for cat in ['high', 'low', 'medium', 'mild']:
    (severity_output / cat).mkdir(parents=True, exist_ok=True)
    cat_files = list((ssl_input_severity / cat).glob('*.jpg'))
    print(f"Processing {cat.upper()}: {len(cat_files)} images")
    
    for f in cat_files:
        img = cv2.imread(str(f))
        if img is not None:
            cv2.imwrite(str(severity_output / cat / f.name), apply_medical_sharpening(img))

print("\n✅ --- DOMAIN ALIGNMENT COMPLETE ---")

🚀 Sharpening 5870 General Knowledge images...
Done: 1000
Done: 2000
Done: 3000
Done: 4000
Done: 5000

🚀 Sharpening Severity categories...
Processing HIGH: 1000 images
Processing LOW: 1000 images
Processing MEDIUM: 1000 images
Processing MILD: 1000 images

✅ --- DOMAIN ALIGNMENT COMPLETE ---


In [6]:
import shutil
import os

# Define the folder you want to zip and the name of the output zip file
folder_to_zip = '/kaggle/working/final_ssl_aligned'
output_zip = '/kaggle/working/final_ssl_aligned'

# Create the zip file
if os.path.exists(folder_to_zip):
    shutil.make_archive(output_zip, 'zip', folder_to_zip)
    print(f"✅ Success! Your dataset has been zipped: {output_zip}.zip")
    print("You can now download it from the 'Output' section on the right sidebar.")
else:
    print(f"❌ Error: The folder {folder_to_zip} was not found. Did the cropping script finish?")

✅ Success! Your dataset has been zipped: /kaggle/working/final_ssl_aligned.zip
You can now download it from the 'Output' section on the right sidebar.


In [8]:
import cv2
import os
import numpy as np
from pathlib import Path

def apply_medical_sharpening(img):
    # 1. Noise Suppression (Bilateral Filter)
    img = cv2.bilateralFilter(img, d=5, sigmaColor=75, sigmaSpace=75)
    
    # 2. Targeted Contrast Enhancement in LAB Space
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(4,4))
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    cl = clahe.apply(l)
    
    limg = cv2.merge((cl, a, b))
    return cv2.cvtColor(limg, cv2.COLOR_LAB2BGR)

# --- CORRECTED PATHS ---
# Input: Where your data is now
ssl_input_severity = Path("/kaggle/input/ssl-dataset/ssl_severity_dataset")
ssl_input_general = Path("/kaggle/input/ssl-dataset/ssl_eye_dataset")

# Output: Where you want the final, sharp data to go
final_output_base = Path("/kaggle/working/final_sharpened_data")

# --- PROCESS SEVERITY DATASET ---
severity_output = final_output_base / "severity"
print(f"🚀 Sharpening Severity categories...")

for cat in ['high', 'low', 'medium', 'mild']:
    (severity_output / cat).mkdir(parents=True, exist_ok=True)
    cat_files = list((ssl_input_severity / cat).glob('*.jpg'))
    print(f"Processing {cat.upper()}: {len(cat_files)} images")
    
    for f in cat_files:
        img = cv2.imread(str(f))
        if img is not None:
            cv2.imwrite(str(severity_output / cat / f.name), apply_medical_sharpening(img))

# --- PROCESS GENERAL DATASET (Required for SSL Phase 1) ---
general_output = final_output_base / "general"
general_output.mkdir(parents=True, exist_ok=True)
general_files = list(ssl_input_general.glob('*.jpg'))

print(f"\n🚀 Sharpening {len(general_files)} General images for SSL...")
for i, f in enumerate(general_files):
    img = cv2.imread(str(f))
    if img is not None:
        cv2.imwrite(str(general_output / f.name), apply_medical_sharpening(img))
    if (i+1) % 1000 == 0: print(f"Done: {i+1}")

print("\n✅ --- DATASET SHARPENING COMPLETE ---")

🚀 Sharpening Severity categories...
Processing HIGH: 1000 images
Processing LOW: 1000 images
Processing MEDIUM: 1000 images
Processing MILD: 1000 images

🚀 Sharpening 5870 General images for SSL...
Done: 1000
Done: 2000
Done: 3000
Done: 4000
Done: 5000

✅ --- DATASET SHARPENING COMPLETE ---
